# aDDM Tutorial

This notebook showcases the implementation of a modern aDDM, compatible with PyDDM.

### Load the data

In [1]:
from ast import literal_eval
import pandas as pd

# 1. Load data
df_raw = pd.read_csv('1ms_trial_data.csv')

# 2. Drop nuisance trials
to_drop = pd.read_csv("dropped_trials.csv").rename(columns={"parcode": "sub_id"})

df = df_raw.loc[
    ~df_raw.set_index(["sub_id", "trial"]).index.isin(
        to_drop.set_index(["sub_id", "trial"]).index
    )
    & (~df_raw["hidden"])
]

# 3. Adjustments
df['RT'] = (df['RT']*1000).astype(int) # RT unit scaling
df['fixation'] = df['fixation'].apply(literal_eval) # String to list as a result of csv saving
df['choice'] = df['choice'].replace({"left": 0, "right": 1}) # Map choice to 0 or 1

/var/folders/59/03h51wmn6xn1jhvb7kj9fjc40000gn/T/ipykernel_22516/4055317967.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['RT'] = (df['RT']*1000).astype(int) # RT unit scaling
/var/folders/59/03h51wmn6xn1jhvb7kj9fjc40000gn/T/ipykernel_22516/4055317967.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['fixation'] = df['fixation'].apply(literal_eval) # String to list as a result of csv saving
/var/folders/59/03h51wmn6xn1jhvb7kj9fjc40000gn/T/ipykernel_22516/4055317967.py:20: FutureWarning: Down

### Simulating data from empiricals

In [2]:
from simulation import get_corrected_empirical_distributions
import numpy as np

# Make empirical distributions
# value_diffs = np.arange(-4, 4.25, 0.25)
value_diffs = np.unique(df['avgWTP_left'] - df['avgWTP_right'])
legend = {
    "left": {1},
    "right": {2},
    "transition": {0}, 
    "blank_fixation": {4}
}
fixation_col = 'fixation'
left_value_col = 'avgWTP_left'
right_value_col = 'avgWTP_right'

empirical_distributions = get_corrected_empirical_distributions(
    df,
    value_diffs=value_diffs,
    legend=legend,
    fixation_col=fixation_col,
    left_value_col=left_value_col,
    right_value_col=right_value_col,
    cutoff=0.9
)

In [3]:
from simulation import generate_fixations

# Create sample trial conditions
dt = 0.01
seed = 42

trials = df.loc[
    (df['sub_id'] == 304) & (df['trial'] % 2 == 1),
    ['avgWTP_left', 'avgWTP_right']
].copy()
trials['fixation'] = None

rng = np.random.default_rng(seed)
trials_dict = []
for idx, r in trials.iterrows():
    fx = generate_fixations(
        dt, 
        r.avgWTP_left - r.avgWTP_right, 
        empirical_distributions,
        rng=rng
    )
    if fx is not None:
        trials_dict.append({
            "avgWTP_left": r.avgWTP_left,
            "avgWTP_right": r.avgWTP_right,
            "fixation": fx
        })

In [4]:
from simulation import simulate
import pyddm

model_conditions = {'drift_rate': 0.3, 'theta': 0.5, 'noise': 0.6}

results_df = simulate(dt, model_conditions, trials_dict, seed=seed, save_results=False)
# results_df['sub_id'] = f'seed{seed}_subjects{size}_sim'
# results_df['trial'] = range(1, len(trials_clean) + 1)
# results_df = results_df.rename(columns={'fixation': 'fix_sequence'})
results_df = results_df.drop(columns = ['trajectory'])

sample = pyddm.Sample.from_pandas_dataframe(
    results_df,
    choice_column_name="choice",
    rt_column_name="RT",
    choice_names=("left", "right")
)

print(f'Average RT: {results_df["RT"].mean():.2f} seconds (out of {len(results_df)} trials)')
results_df.head()

Average RT: 2.18 seconds (out of 100 trials)


,avgWTP_left,avgWTP_right,fixation,RT,choice
0,1.00,1.00,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",4.09,1
1,4.25,3.00,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",4.82,1
2,3.25,3.75,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, ...",3.17,0
3,3.00,2.75,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",5.37,1
4,1.00,4.00,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.54,0


The above is the first half of the tutorial. Following is native parameter recovery by differential evolution.

In [6]:
# Define the model
def drift_function(avgWTP_left, avgWTP_right, fixation, d, x, t):
        fixation_index = min(int(t/dt), len(fixation)-1)
        current_fixation = fixation[fixation_index]
        if current_fixation == 0: # saccade
            drift_val = 0
        elif current_fixation == 1: # left
            drift_val = d * (avgWTP_left - avgWTP_right * model_conditions['theta'])
        else: # right
            drift_val = d * (avgWTP_left * model_conditions['theta'] - avgWTP_right)
        
        return np.ones_like(x) * drift_val
    
def noise_function(n, x, t):
    return np.ones_like(x) * n

model = pyddm.gddm(
    drift=drift_function,
    noise=noise_function,
    bound=1,
    nondecision=0,
    parameters={'d': (0.1, 0.4), 'n': (0.5, 0.7)},
    conditions=["avgWTP_left", "avgWTP_right", "fixation"],
    choice_names=("left", "right"),
    T_dur=30,
    dx=0.01,
    dt=dt
)

model._overlay = pyddm.models.OverlayChain(overlays=[])

model.fit(sample=sample, verbose=True)

Info: Model(name='n', drift=DriftEasy(d=Fitted(0.28979718453752323, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6853984722282556, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=313.24526328725597
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.23671141602942308, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.5621823788695437, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=326.82311958348856
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.32824090172540454, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.5435554870986754, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=382.42949113040794
I

differential_evolution step 1: f(x)= 256.92104861200613


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.12209164030278732, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6496059310420609, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=263.34733027568666
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.15748612363779435, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6797186922685228, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=270.9838043265942
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.3210203825748076, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6734719139794305, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=327.0557413721231
Info

differential_evolution step 2: f(x)= 256.92104861200613


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.24305796249856776, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6943111058428448, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=296.0719139737003
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.16150955603497516, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.652169456031533, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=274.840488791476
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.3210203825748076, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6096439274361863, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=345.5047989568359
Info: M

...

Info: Model(name='n', drift=DriftEasy(d=Fitted(0.11115511662828995, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6777118158137752, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=258.31514254010295
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.13464789975365538, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6712384524286092, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=265.0755337499779
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.11013459996662994, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6101930065302152, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=265.09759228122675
In

differential_evolution step 8: f(x)= 255.32010081623855


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.35679828616693393, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6989849672948965, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=335.94348838019425
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.10284532868836255, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.631417900432146, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=259.8221534838631
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.11120605751259438, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.574593668593886, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=273.033048458051
Info: 

differential_evolution step 9: f(x)= 255.11928915874688
Polishing solution with 'L-BFGS-B'


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.10058527130305983, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6965669697211664, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=255.11928915874688
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.10058528130305983, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6965669697211664, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=255.1192916251642
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.10058527130305983, minval=0.1, maxval=0.4)), noise=NoiseEasy(n=Fitted(0.6965669797211664, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=255.11928906695735
In